## VQA는 무엇을 반복해서 바꾸는가?
- 우리말로 변분 양자 알고리즘(Variational Qunatum Algoritm)
- VQA 를 이해할 때는 문제의 평가 기준, 후보 상태를 만드는 회로, 각도를 조정하는 최적화기를 함께 살펴본다.
-> 평가 기준은 무엇이 좋은 답인지 정하고, 회로는 어떤 후보 상태를 만들 수 있는지, 최적화기는 지금까지 계산한 평가값을 바탕으로 다음에 사용할 각도를 선택
- VQE, QAOA 는 이 구조를 구체화한 알고리즘이다.

## VQE 작동 원리
- 개념적이해 : VQE는 현재 노이즈가 많은 양자 컴퓨터와 고전 컴퓨터가 협력하여 분자 물질의 가장 낮은 에너지(안정된 상태)를 찾고 복잡한 최적화 문제를 해결하는데 사용되는 실용적인 하이브리드 알고리즘이다.
- 구체적으로 VQE는 양자 회로(Quantum Circuit)와 고전적 최적화(Classical Optimization) 를 결합하여 주어진 해밀토니안의 최소 고유값(ground state energy)를 찾는 하이브리드 알고리즘이다.
- 양자 장비에서는 회로로 상태를 준비하고 관측량을 측정한다. 고전 컴퓨터는 측정 결과를 합산하여 평가값을 계산하고 그 평가값을 개선할 다음 매개변수를 선택한다.

-> 낮은 에너지를 찾는다는 의미
- VQE의 입력은 문제를 표현한 해밀토니안과 매개변수화된 회로이다. 출력은 최적화 과정에서 찾은 매개변수와 그 상태의 에너지이다. 해밀토니안은 상태마다 어떤 에너지를 부여할지 정하는 연산자이다. 가장 낮은 고유값은 해당 모형에서 가능한 최소에너지에 해당한다.
- 회로에 처음 넣은 각도가 좋은 값을 만든다는 보장은 없다. VQE는 첫 각도에서 에너지를 계산하고 다른 각도로 다시 계산하는 과정을 반복한다. 이때 해밀토니안과 회로의 기본 구조는 정해 둔 상태에서 회전각을 주로 조정한다. 정해진 회로가 표현할 수 있는 상태 범위 안에서 에너지가 작은 상태를 찾는 것

## VQE 내용
1. Variational Principle
- 문자식 해밀토니안 관련, 각도 세타로 만든 상태의 평균 에너지를 계산한다.

## 안자츠(Ansatz) 회로는 매개 변수 벡터 세타에 따라 '후보 양자 상태'를 생성하는 역할을 하며, 해밀토니안은 그 상태 기대 에너지를 계산하여, 에너지가 최소가 되도록 매개변수를 최적화하는 평가 기준으로 작동한다.

## Ansatz를 회전 손잡이로 이해하기
- Ansatz는 후보 상태를 만드는 회로의 틀이다. Ry 게이트의 각도는 상태의 진폭을 바꾸는 조절값입니다. CNOT은 두 큐비트의 상태를 연결하며, 입력 상태에 따라 얽힘을 만들 수 있다. 뒤에 배치한 Ry는 연결된 상태를 다시 조정한다.
- 회로의 층을 늘리면 표현할 수 있는 상태가 다양해질 수 있다. 함께 늘어나는 매개변수와 회로 실행량도 고려해야 한다. 실제 장비에서는 깊어진 회로가 잡음에 더 많이 노출될 수 있다. 따라서 층을 늘리는 결정은 에너지 오차와 계산 부담을 함께 확인하면서 내린다.

## 대표적인 옵티마이저
- COBYLA (gradient-free)
- SPSA (확률적 근사 기울기)
- Nelder–Mead

## COBYLA (gradient-free)
- Gradient-Free (기울기 불필요)
- 작동방식 : 주변의 몇 개 지점만 탐색하여 가장 낮은 곳으로 이동한다. '주변을 둘러보고 가장 낮은 언덕으로 한 걸음씩' 내려가는 방식이다.
- 주요 장점 및 사용처 : 제약 조건(Constraints) 설정이 용이. VQE에서 가장 널리 쓰이는 표준 옵티마이저이다.

## SPSA
- 확률적 근사 기울기 (Stochastic Approximation)
- 작동방식 : 정확한 기울기를 계산하는 대신, 단 두 번의 무작위 측정으로 기울기를 '대충' 파악하여 이동한다.
- 주요 장점 및 사용처 : 대규모 또는 노이즈가 많은 시스템에 적합. 계산 비용이 매우 저렴하여, 측정이 느린 양자 시스템에서 효율적이다.

## Nelder–Mead
- Gradient-Free / 심플렉스(Simplex) 기반
- 작동방식 : 삼각형(2D) 또는 피라미드(3D 이상) 모양의 탐색 영역을 만들고, 가장 높은 지점을 제거한 후 새로운 지점을 생성하며 점차 가장 낮은 바닥으로 모양을 좁혀나간다.
- 주요 장점 및 사용처 : 미분 없이 사용할 수 있다. 초기 심플렉스, 문제의 차원, 잡음에 따라 수렴 결과가 달라지며 전역 최적해를 보장하지 않는다.

## 수렴 (Convergence)
수렴은 평가값이나 매개변수의 변화가 정해 둔 기준보다 작아졌다는 뜻이다. 최대 평가 횟수에 도달해서 계산이 종료될 수도 있다. 종료 사실만으로 전역 최적해를 찾았다고 판단하지 않는다. 작은 문제에서는 정확한 고유값 계산 결과와 비교하여 오차를 확인한다.

최적화로 얻은 θ*를 회로에 넣으면 최종 후보 상태를 다시 만들 수 있다. 에너지가 실제 최소값에 충분히 가까운지, 초기 각도를 바꿨을 때도 비슷한 결과가 나오는지 확인한다. 에너지 오차만으로 상태 자체의 오차를 판단하려면 에너지 간격과 기저 상태의 중복 여부도 고려해야 한다.

## 파이프라인

## QAOA 작동 원리

### 개념적 이해

**QAOA (Quantum Approximate Optimization Algorithm)** 는 양자 회로를 이용해 **조합최적화 (Combinatorial Optimization)** 문제를 근사적으로 푸는 알고리즘이다.

선택을 0과 1로 표시하면 여러 선택의 조합을 하나의 비트열로 표현할 수 있다. 예를 들어 네 지점을 두 그룹으로 나누는 문제에서 0101은 각 지점을 어느 그룹에 넣을지를 나타낸다. 비트열에 비용을 부여한 뒤, 비용이 작은 비트열을 찾는 문제로 구성한다.

QAOA는 **VQE와 유사한 변분(Variational) 접근**을 사용하지만, 문제에 따라 특별히 설계된 **두 종류의 해밀토니안(Hamiltonian)** 을 번갈아 적용한다는 점이 핵심이다.

## QAOA 내용

### 2.1 비트열에 비용을 부여하기

QAOA를 실행하기 전에 각 비트가 무엇을 의미하는지 정해야 한다. 그룹 배정 문제에서는 그룹 번호를 나타낼 수 있고, 일정 문제에서는 특정 작업을 특정 시간에 배정하는지를 나타낼 수 있다. 같은 0011이라도 문제 정의에 따라 의미가 달라진다.

해밀토니안은 비트열을 평가하는 규칙을 회로 계산에 사용할 수 있는 형태로 표현한다. 최소화 문제에서는 좋은 답에 작은 비용을 부여한다. MaxCut처럼 값을 최대화하려는 문제에서는 비용의 부호를 바꾸어 최소화 문제로 변환할 수 있다. 이 부호가 잘못되면 최적화기는 의도와 다른 답을 찾는다.

(1) 목적함수 → 해밀토니안으로 변환  : 설명 - 개별 선택의 비용과 두 선택 사이의 관계를 연산자로 표현하여 합한다. 이 식은 일차·이차 항으로 표현되는 비용의 예다. 일반적인 비용에는 상수항이나 더 많은 Z를 곱한 항이 필요할 수 있으며, 뒤의 경로 예제는 후보별 비용을 대각행렬에 직접 넣는다.
(2) 보조 해밀토니안 (Mixer Hamiltonian) : 설명 - 각 큐비트에 X 연산자 항을 두어, 비용 연산과 번갈아 사용할 믹서 해밀토니안을 만든다.

### 2.2 γ와 β가 조절하는 것

γ는 비용 해밀토니안의 작용 정도를 조절한다. 비용이 다른 상태들은 서로 다른 위상 변화를 얻는다. 대각 비용 연산만 적용한 직후에는 각 비트열의 측정 확률이 그대로일 수 있다. 뒤이어 믹서를 적용하면 상대 위상 차이가 진폭의 간섭에 영향을 주고, 측정 확률도 달라진다.

β는 믹서의 작용 정도를 조절한다. 두 연산의 조합을 하나의 층으로 보고 반복 횟수를 p로 표시한다. p가 2라면 표준 QAOA에서 두 개의 γ와 두 개의 β를 조정한다. 회로 깊이가 늘어나면 표현력이 커질 수 있으나, 최적화 난이도와 실행 비용도 함께 달라진다.

(3) 파라미터화된 양자 상태 : 설명 - 오른쪽의 초기 상태에서 출발한다. 비용 연산과 믹서 연산을 차례로 적용하고, 각 층의 각도를 조절하여 측정 결과의 확률 분포를 바꾼다.

### 2.3 상태 수식의 읽는 순서

양자 상태 수식은 오른쪽에서 왼쪽으로 적용 순서를 읽는다. 먼저 균등 중첩 상태를 만들고, 첫 번째 비용 연산과 첫 번째 믹서를 적용한다. 이어서 다음 층의 비용 연산과 믹서를 적용한다. 지수함수 표기는 해밀토니안이 정해진 크기만큼 작용하여 상태를 변화시키는 유니터리 연산을 의미한다.

이 구조에서 고전 최적화기가 직접 바꾸는 것은 γ와 β이다. 그래프의 간선이나 배송 거리처럼 입력으로 정한 문제 데이터는 그대로 유지된다. 각도를 바꿀 때마다 회로의 출력 분포가 달라지고, 그 분포에서 계산한 평균 비용을 다시 평가한다.

(4) 기대값 최소화 : 설명 - 현재 각도에서 여러 후보가 나올 확률을 고려한 평균 비용을 구한다. 가장 좋은 후보 하나의 비용과 이 평균값은 서로 다를 수 있다.

### 3.1 MaxCut 결과 해석

기본 QAOA 코드에서는 네 꼭짓점과 다섯 간선을 사용한다. 서로 다른 그룹에 속한 꼭짓점 사이의 간선 수를 Cut이라고 부른다. 각 비트는 꼭짓점의 그룹 번호다. 코드의 −Cut 해밀토니안을 최소화하면 Cut이 커지는 방향으로 탐색한다.

실행 예에서는 선택한 비트열은 1010이고 Cut은 4다. 전수 탐색의 최댓값도 4이므로 선택한 후보는 이 그래프의 최적해다. 분포의 평균 에너지는 약 −3.428223이었고, 최적 Cut을 갖는 후보들의 관측 확률 합은 약 43.41%다. 한 번의 측정으로 매번 최적해가 나오는 상황은 아니며, 여러 후보를 관측하여 평가하는 과정이 필요하다.

StatevectorSampler의 default_shots=2048은 해당 회로 평가에서 2,048개의 측정 표본을 생성하도록 설정한 값이다. 시드는 재현성을 돕는다. 막대그래프의 가로축은 비트열이고 세로축은 관측 비율dl다. 비트열 문자열은 2진수로 변환해야 경로 번호나 상태 번호와 정확히 대응된다.

## QAOA는
“문제의 비용함수(Cost Hamiltonian)”와 “탐색을 위한 Mixer Hamiltonian”을 교대로 적용하면서, 파라미터 (γ, β)를 조정해 최적 비트 조합을 근사적으로 찾는 양자 알고리즘